# Librerias

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer
from category_encoders import TargetEncoder


# Lectura de datos

In [2]:
df_train = pd.read_csv("../data/raw/df_train.csv")
df_test = pd.read_csv("../data/raw/df_test.csv")

C:\Users\Administrador\AppData\Local\Temp\ipykernel_21396\3809680416.py:1: DtypeWarning: Columns (0: type_y, 1: locale, 2: locale_name, 3: description, 4: transferred) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv("../data/raw/df_train.csv")
C:\Users\Administrador\AppData\Local\Temp\ipykernel_21396\3809680416.py:2: DtypeWarning: Columns (0: type_y, 1: locale, 2: locale_name, 3: description, 4: transferred) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv("../data/raw/df_test.csv")


In [3]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 2642706 entries, 0 to 2642705
Data columns (total 17 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            int64  
 1   date          str    
 2   store_nbr     int64  
 3   family        str    
 4   sales         float64
 5   onpromotion   int64  
 6   city          str    
 7   state         str    
 8   type_x        str    
 9   cluster       int64  
 10  transactions  float64
 11  dcoilwtico    float64
 12  type_y        str    
 13  locale        str    
 14  locale_name   str    
 15  description   str    
 16  transferred   object 
dtypes: float64(3), int64(4), object(1), str(9)
memory usage: 342.8+ MB


In [4]:
df_train.rename(columns={'type_x': 'type_store', 'type_y': 'type_holiday', 'dcoilwtico': 'oil_price'}, inplace=True)
df_test.rename(columns={'type_x': 'type_store', 'type_y': 'type_holiday', 'dcoilwtico': 'oil_price'}, inplace=True)

In [5]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 2642706 entries, 0 to 2642705
Data columns (total 17 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            int64  
 1   date          str    
 2   store_nbr     int64  
 3   family        str    
 4   sales         float64
 5   onpromotion   int64  
 6   city          str    
 7   state         str    
 8   type_store    str    
 9   cluster       int64  
 10  transactions  float64
 11  oil_price     float64
 12  type_holiday  str    
 13  locale        str    
 14  locale_name   str    
 15  description   str    
 16  transferred   object 
dtypes: float64(3), int64(4), object(1), str(9)
memory usage: 342.8+ MB


In [6]:
#Comprobación de que están ordenadas.
start_tr = df_train['date'].head().tolist()
end_tr = df_train['date'].tail().tolist()
start_tt = df_test['date'].head().tolist()
end_tt = df_test['date'].tail().tolist()

print(f'TRAIN | start: {start_tr} ; end: {end_tr}')
print(f'TEST | start: {start_tt} ; end: {end_tt}')

TRAIN | start: ['2013-01-01', '2013-01-01', '2013-01-01', '2013-01-01', '2013-01-01'] ; end: ['2016-12-31', '2016-12-31', '2016-12-31', '2016-12-31', '2016-12-31']
TEST | start: ['2017-01-01', '2017-01-01', '2017-01-01', '2017-01-01', '2017-01-01'] ; end: ['2017-08-15', '2017-08-15', '2017-08-15', '2017-08-15', '2017-08-15']


# Selección de Características

In [7]:
# Todas lo que escrito en la lista es lo que nos eliminaremos al final, una vez hayamos hecho las transformaciones, el feature engineering, etc.
cols_drop = ['id','store_nbr','date','city','locale','locale_name','state','description','transferred','type_holiday']

## Gestión de tipos

In [8]:
df_train['date'] = pd.to_datetime(df_train['date'])
df_test['date'] = pd.to_datetime(df_test['date'])

In [9]:
df_train['cluster']=df_train['cluster'].astype('object')
df_test['cluster']=df_test['cluster'].astype('object')

## Feature Engineering

In [10]:
def extract_date_features(df):
    """ Realiza ingeniería de características (feature engineering) sobre una columna de fechas.
    
    Extrae componentes temporales básicos y crea variables binarias de contexto 
    específicas para el análisis de ventas (fines de semana, estacionalidad de diciembre, 
    días de pago y eventos sísmicos).
    
    Args:
        df (pd.DataFrame): DataFrame que debe contener obligatoriamente una 
                           columna llamada 'date' en formato datetime.
                           
    Returns:
        pd.DataFrame: El DataFrame original con las siguientes columnas adicionales:
            - year (int): Año de la observación.
            - month (int): Mes del año (1-12).
            - day (int): Día del mes (1-31).
            - day_week (int): Día de la semana (0=Lunes, 6=Domingo).
            - is_weekend (int): 1 si es Sábado o Domingo, 0 en caso contrario.
            - is_christmas (int): 1 si el mes es Diciembre, 0 en caso contrario.
            - es_pago (int): 1 si es día 15 o último día del mes, 0 en caso contrario.
            - terremoto (int): 1 si la fecha es igual o posterior al 16 de abril de 2016."""

    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['day_week'] = df['date'].dt.dayofweek # 0=Lunes, 6=Domingo
    
    # is_weekend: Sábado (5) y Domingo (6)
    # Vimos que la media subía de 304 a 427, ¡es clave!
    df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)
    
    # is_christmas (Diciembre):
    # Según tu gráfico image_b30700.png, diciembre es el pico máximo
    df['is_christmas'] = (df['date'].dt.month == 12).astype(int)
    
    # Día de pago (Quincenas)
    df['es_pago'] = ((df['date'].dt.day == 15) | df['date'].dt.is_month_end).astype(int)
    
    # Terremoto (Variable de choque)
    df['terremoto'] = (df['date'] >= '2016-04-16').astype(int)

    #Creamos variable is_workday
    laboral = ['Work Day']

    #Es laboral (1) si el tipo es 'Work Day' O si el festivo fue transferido (transferred == True)
    df['is_workday'] = (df['type_holiday'].isin(laboral) | (df['transferred'] == True)).astype(int)
   
    return df

df_train = extract_date_features(df_train)
df_test = extract_date_features(df_test)

df_train.shape, df_test.shape

((2642706, 26), (411642, 26))

In [11]:
# Aplicamos la eliminación y guardamos el resultado
df_train = df_train.drop(columns=cols_drop)
df_test = df_test.drop(columns=cols_drop)

# Verificación rápida del "equipo final" de columnas
print(f"Columnas finales en Train ({len(df_train.columns)}): {df_train.columns.tolist()}")

Columnas finales en Train (16): ['family', 'sales', 'onpromotion', 'type_store', 'cluster', 'transactions', 'oil_price', 'year', 'month', 'day', 'day_week', 'is_weekend', 'is_christmas', 'es_pago', 'terremoto', 'is_workday']


# Gestión de variables categóricos

In [12]:
categoric_cols = df_train.select_dtypes(include='object').columns
categoric_cols

C:\Users\Administrador\AppData\Local\Temp\ipykernel_21396\2161281752.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoric_cols = df_train.select_dtypes(include='object').columns


Index(['family', 'type_store', 'cluster'], dtype='str')

In [13]:
# Vamos a agrupar en family en tres tipos de categorias: Alimentos, despensa y otros. Para reducir la cardinalidad

#Definición del diccionario de mapeo (basado en las 33 categorías originales)
family_mapping = {
    # ALIMENTOS_FRESCOS
    'MEATS': 'ALIMENTOS_FRESCOS', 'POULTRY': 'ALIMENTOS_FRESCOS', 
    'SEAFOOD': 'ALIMENTOS_FRESCOS', 'PRODUCE': 'ALIMENTOS_FRESCOS', 
    'DAIRY': 'ALIMENTOS_FRESCOS', 'DELI': 'ALIMENTOS_FRESCOS', 'EGGS': 'ALIMENTOS_FRESCOS',

    #ABARROTES_DESPENSA
    'GROCERY I': 'ABARROTES_DESPENSA', 'GROCERY II': 'ABARROTES_DESPENSA', 
    'BEVERAGES': 'ABARROTES_DESPENSA', 'BREAD/BAKERY': 'ABARROTES_DESPENSA', 
    'FROZEN FOODS': 'ABARROTES_DESPENSA', 'PREPARED FOODS': 'ABARROTES_DESPENSA'
}

#Aplicar la transformación de forma permanente
#Usamos .fillna('OTROS') para agrupar las 20 categorías restantes[cite: 1]
df_train['family'] = df_train['family'].map(family_mapping).fillna('OTROS')
df_test['family'] = df_test['family'].map(family_mapping).fillna('OTROS')

#Verificación de la reducción de cardinalidad[cite: 1]
print(f"Nuevas categorías únicas en family: {df_train['family'].unique()}")

Nuevas categorías únicas en family: <StringArray>
['OTROS', 'ABARROTES_DESPENSA', 'ALIMENTOS_FRESCOS']
Length: 3, dtype: str


In [14]:
# 1. Definimos qué columnas van a cada transformador
cols_ohe = ['family', 'type_store']
cols_target = ['cluster']

# 2. Creamos el ColumnTransformer
# 'remainder=passthrough' asegura que las columnas que NO mencionamos 
# (como month, is_weekend, etc.) se queden tal cual en el dataframe.
preprocessor = ColumnTransformer(
    transformers=[
        # One-Hot Encoding para variables con pocas categorías
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cols_ohe),
        
        # Target Encoding para 'cluster'
        ('cat_target', TargetEncoder(smoothing=10), cols_target)
    ],
    remainder='passthrough'
)

# 3. Creamos la Pipeline final
# Aquí podrías añadir después un escalador (StandardScaler) o el modelo directamente
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

# 4. Ajustar y transformar los datos
# IMPORTANTE: El Target Encoding necesita 'y' (sales) para aprender las medias
X_train_encoded = model_pipeline.fit_transform(df_train.drop(columns=['sales']), df_train['sales'])

# Para el test, solo transformamos (usando las medias aprendidas en train)
X_test_encoded = model_pipeline.transform(df_test)

# Missing Values

In [15]:
(df_train.isnull().sum() / df_train.shape[0])*100

family           0.000000
sales            0.000000
onpromotion      0.000000
type_store       0.000000
cluster          0.000000
transactions     9.224295
oil_price       31.018206
year             0.000000
month            0.000000
day              0.000000
day_week         0.000000
is_weekend       0.000000
is_christmas     0.000000
es_pago          0.000000
terremoto        0.000000
is_workday       0.000000
dtype: float64

In [16]:
def impute_transactions(df):
    """
    Realiza la imputación de valores nulos y la transformación de potencia 
    (Yeo-Johnson) para la columna 'transactions'.
    
    La imputación sigue una lógica de negocio: si las ventas son 0, se asume que 
    la tienda estaba cerrada y se imputa 0 en transacciones. En caso contrario, 
    se utiliza la mediana del conjunto de entrenamiento para evitar el sesgo 
    de los outliers identificados en el EDA.
    
    Args:
        df_train (pd.DataFrame): Conjunto de datos de entrenamiento que contiene 
                                 las columnas 'transactions' y 'sales'.
        df_test (pd.DataFrame): Conjunto de datos de prueba/validación.
                                 
    Returns:
        tuple: (df_train, df_test) con la columna 'transactions' procesada.
    """
    
    # Calculamos la mediana del set de entrenamiento para evitar data leakage
    median_transactions = df['transactions'].median()
    
    # Caso A: Si sales es 0 y transactions es NaN -> Imputamos 0
    df.loc[(df['transactions'].isna()) & (df['sales'] == 0), 'transactions'] = 0
    
    # Caso B: Resto de NaN (tiendas abiertas sin registro) -> Imputamos la mediana
    df['transactions'] = df['transactions'].fillna(median_transactions)
    
    return df

# Aplicamos la imputación a train y test
df_train = impute_transactions(df_train)
df_test = impute_transactions(df_test)

In [20]:
def impute_oil(df):
    """
    Imputa los valores nulos del precio del petróleo usando interpolación lineal,
    ya que los nulos corresponden a fines de semana y festivos donde no hay cotización.
    """
    # 1. Aseguramos que los datos estén ordenados por fecha (muy importante para interpolar)
    df = df.sort_values(by=['store_nbr', 'date'])

#   Interpolación lineal para rellenar los huecos del fin de semana
    df['oil_price'] = df['oil_price'].interpolate(method='linear')

    #ffill() y bfill() por si el primer o último registro del dataset es nulo
    df['oil_price'] = df['oil_price'].ffill().bfill()

    return df

#Aplicamos la función a Train y Test
df_train = impute_oil(df_train)
df_test = impute_oil(df_test)

#Verificación
print(f"Nulos en oil_price (Train): {df_train['oil_price'].isna().sum()}")

KeyError: 'store_nbr'

# Outliers

In [17]:
print("--- Resumen de Outliers ---")
for col in df_train.select_dtypes(exclude='object'):
    Q1 = df_train[col].quantile(0.25)
    Q3 = df_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_train[(df_train[col] < lower_bound) | (df_train[col] > upper_bound)]
    perc = (len(outliers) / len(df_train)) * 100
    print(f"{col}: {len(outliers)} registros ({perc:.2f}%)")

--- Resumen de Outliers ---
sales: 395096 registros (14.95%)
onpromotion: 439508 registros (16.63%)
transactions: 141009 registros (5.34%)
oil_price: 0 registros (0.00%)
year: 0 registros (0.00%)
month: 0 registros (0.00%)
day: 0 registros (0.00%)
day_week: 0 registros (0.00%)
is_weekend: 0 registros (0.00%)
is_christmas: 222750 registros (8.43%)
es_pago: 171072 registros (6.47%)
terremoto: 481140 registros (18.21%)
is_workday: 17820 registros (0.67%)


In [ ]:
log_cols = ['onpromotion', 'sales']

# Sobreescribe las columnas originales con su versión logarítmica
df_train[log_cols] = np.log1p(df_train[log_cols])

In [ ]:
print("--- Resumen de Outliers ---")
for col in df_train.select_dtypes(exclude='object'):
    Q1 = df_train[col].quantile(0.25)
    Q3 = df_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_train[(df_train[col] < lower_bound) | (df_train[col] > upper_bound)]
    perc = (len(outliers) / len(df_train)) * 100
    print(f"{col}: {len(outliers)} registros ({perc:.2f}%)")

# Normalización/ Estandarización

In [ ]:
pt = PowerTransformer(method="yeo-johnson")

# Es importante ajustar (fit) solo con los datos de entrenamiento
df_train['transactions'] = pt.fit_transform(df_train[['transactions']])

# Y transformar (transform) los datos de test con los parámetros aprendidos
df_test['transactions'] = pt.transform(df_test[['transactions']])

print("Nulos en transactions tras imputación:", df_train['transactions'].isna().sum())

# Exportación de datos